Celda 1 - Instalación

In [ ]:
!pip install -q markitdown[all] pdfplumber pymupdf pymupdf4llm pytesseract Pillow pandas openpyxl python-docx python-pptx rich tqdm
!apt-get install -q tesseract-ocr tesseract-ocr-spa tesseract-ocr-eng

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472

Celda 2 - imports

In [ ]:
import os
import json
import time
import zipfile
from pathlib import Path
from datetime import datetime

from markitdown import MarkItDown
import pdfplumber
import fitz
import pymupdf4llm
import pytesseract
from PIL import Image
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich.progress import track

console = Console()
md_converter = MarkItDown(enable_plugins=False)

Celda 3 - Configuración

In [ ]:
OCR_LANG = "spa+eng"
ROOT = Path("/content")
SUPPORTED_EXTENSIONS = {
    ".pdf", ".docx", ".doc", ".xlsx", ".xls",
    ".pptx", ".ppt", ".csv", ".json", ".xml",
    ".html", ".htm", ".txt", ".md",
    ".png", ".jpg", ".jpeg", ".webp", ".tiff", ".bmp",
    ".epub", ".zip"
}

files_found = [
    f for f in ROOT.iterdir()
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
]

console.print(f"\n📄 Encontrados {len(files_found)} archivo(s):")
for f in files_found:
    mb = round(f.stat().st_size / 1024 / 1024, 1)
    console.print(f"   • {f.name} ({mb} MB)")

📄 Encontrados 2 archivo(s):

• 4.8_     P8.2.2,,1 Auditorias.pdf (0.4 MB)

• 4 10_    P7 4_compras, ok oscar.pdf (0.4 MB)

Celda 4 - Funciones de conversion fallback

In [ ]:
def is_pdf_scanned(path):
    doc = fitz.open(path)
    text = "".join(doc[i].get_text() for i in range(min(3, len(doc))))
    doc.close()
    return len(text.strip()) < 50

def ocr_from_image(img):
    try:
        return pytesseract.image_to_string(img, lang=OCR_LANG).strip()
    except Exception:
        return ""

def convert_pdf(path):
    if is_pdf_scanned(path):
        doc = fitz.open(path)
        pages = []
        for i, page in enumerate(doc):
            pix = page.get_pixmap(dpi=200)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = ocr_from_image(img)
            pages.append(f"## Página {i+1}\n\n{text}")
        doc.close()
        return "\n\n---\n\n".join(pages), "OCR/Tesseract"

    try:
        return pymupdf4llm.to_markdown(path), "pymupdf4llm"
    except Exception:
        pass

    try:
        parts = []
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                parts.append(f"## Página {i+1}\n\n{text.strip()}")
        return "\n\n---\n\n".join(parts), "pdfplumber"
    except Exception:
        pass

    try:
        result = md_converter.convert(path)
        return result.text_content, "markitdown"
    except Exception as e:
        return f"# ERROR\n\n{str(e)}", "ERROR"

def convert_image(path):
    try:
        img = Image.open(path)
        text = ocr_from_image(img)
        if text:
            return f"# {Path(path).name}\n\n{text}", "OCR/Tesseract"
        raise ValueError("OCR vacío")
    except Exception:
        pass

    try:
        result = md_converter.convert(path)
        return result.text_content, "markitdown/image"
    except Exception as e:
        return f"# ERROR\n\n{str(e)}", "ERROR"

def convert_excel(path):
    try:
        ext = Path(path).suffix.lower()
        if ext == ".csv":
            sheets = {"Sheet1": pd.read_csv(path, encoding="utf-8-sig")}
        else:
            xl = pd.ExcelFile(path)
            sheets = {s: xl.parse(s) for s in xl.sheet_names}
        parts = [f"## {name}\n\n{df.to_markdown(index=False)}" for name, df in sheets.items()]
        return "\n\n---\n\n".join(parts), "pandas"
    except Exception:
        pass

    try:
        result = md_converter.convert(path)
        return result.text_content, "markitdown/excel"
    except Exception as e:
        return f"# ERROR\n\n{str(e)}", "ERROR"

def convert_file(path):
    ext = Path(path).suffix.lower()
    try:
        if ext == ".pdf":
            return convert_pdf(path)
        elif ext in {".png", ".jpg", ".jpeg", ".webp", ".tiff", ".bmp"}:
            return convert_image(path)
        elif ext in {".xlsx", ".xls", ".csv"}:
            return convert_excel(path)
        else:
            result = md_converter.convert(path)
            return result.text_content, "markitdown"
    except Exception as e:
        return f"# ERROR\n\n{str(e)}", "ERROR"

Celda 5 - Engine principal

In [ ]:
def to_kebab(name):
    return name.lower().replace(" ", "-").replace("_", "-")

def run():
    if not files_found:
        console.print("[yellow]No hay archivos en /content[/yellow]")
        return

    console.rule(f"[bold]Procesando {len(files_found)} archivos[/bold]")
    results = []
    t_total = time.time()

    for f in track(files_found, description="Convirtiendo..."):
        t0 = time.time()
        content, method = convert_file(str(f))
        elapsed = round(time.time() - t0, 2)

        header = (
            f"---\n"
            f"source: {f.name}\n"
            f"method: {method}\n"
            f"date: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n"
            f"---\n\n"
        )

        out_name = to_kebab(f.stem) + ".md"
        out_file = ROOT / out_name
        out_file.write_text(header + content, encoding="utf-8")

        kb     = round(out_file.stat().st_size / 1024, 1)
        status = "✅" if method != "ERROR" else "❌"
        console.print(f"{status} {f.name} → {out_name} [{method}] {kb}KB {elapsed}s")

        results.append({"file": f.name, "output": out_name, "method": method, "kb": kb, "s": elapsed})

    console.rule("[bold green]DONE[/bold green]")
    console.print(f"⏱️ Total: {round(time.time()-t_total, 2)}s")

    log = ROOT / "_log.json"
    log.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

run()

────────────────────────────────────────────── Procesando 2 archivos ──────────────────────────────────────────────

Output()

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=11/12.
OCR on page.number=12/13.



✅ 4.8_     P8.2.2,,1 Auditorias.pdf → 4.8------p8.2.2,,1-auditorias.md  15.6KB 16.77s

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                   Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.



✅ 4 10_    P7 4_compras, ok oscar.pdf → 4-10-----p7-4-compras,-ok-oscar.md  8.4KB 8.99s

────────────────────────────────────────────────────── DONE ───────────────────────────────────────────────────────

⏱️ Total: 25.77s

Celda 6 - Descarga en ZIP

In [ ]:
from google.colab import files

md_files = list(ROOT.glob("*.md"))
console.print(f"\n📥 Descargando {len(md_files)} archivo(s)...")
for md in md_files:
    files.download(str(md))

📥 Descargando 2 archivo(s)...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>